In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import plot
import re

# CARGA, LIMPIEZA Y ENRIQUECIMIENTO DE DATOS

# Cargar dataset
df = pd.read_csv("pisos_andorra.csv")

print(f"el csv tiene {df.shape[0]} registros")

# Forzar conversión a tipos de datos correctos
df['precio'] = pd.to_numeric(df['precio'], errors='coerce')
df['m2'] = pd.to_numeric(df['m2'], errors='coerce')
df['eur_m2'] = pd.to_numeric(df['eur_m2'], errors='coerce')
df = df.dropna(subset=['precio', 'm2', 'eur_m2', 'municipio', 'tipo_vivienda'])

# Arreglamos tipos de vivienda para agruparlos en categorías más amplias y evitar la dispersión excesiva de tipologías con pocas muestras.
mapeo_viviendas = {
    'Piso': 'Piso en alto',
    'Apartamento': 'Piso en alto',
    'Dúplex': 'Piso en alto',
    'Estudio': 'Piso en alto',
    'Ático': 'Ático',
    'Casa adosada': 'Casa adosada',
    'Chalet adosado': 'Casa adosada',
    'Chalet pareado': 'Casa adosada',
    'Casa pareada': 'Casa adosada',
    'Chalet': 'Chalet',
    'Casa': 'Chalet',
    'Chalet unifamiliar': 'Chalet',
    'Casa unifamiliar': 'Chalet',
    'Casa rústica': 'Vivienda rústica',
    'Finca rústica': 'Vivienda rústica',
    'Chalet rústico': 'Vivienda rústica'
}

# Aplicamos el mapeo
df['tipo_vivienda'] = df['tipo_vivienda'].map(mapeo_viviendas)

# Definimos la función para normalizar los nombres de los municipios, eliminando paréntesis y corrigiendo duplicados comunes.
def normalizar_municipio(texto):
    if pd.isna(texto):
        return texto
    
    # Quitamos espacios
    texto = str(texto).strip()
    
    # Eliminamos los parentesis y su contenido, si existen para quedarnos solo con el nombre principal del municipio
    if '(' in texto:
        texto = texto.split('(')[0].strip()
        
    # Arreglamos manualmente algunos nombres comunes que pueden aparecer con variaciones o errores de tipeo
    mapeo_correcciones = {
        'Centre': 'Andorra la Vella',  # Habitualmente "Centre" se refiere al centro de la capital
        'Escaldes Engordany': 'Escaldes-Engordany',
        'Alàs': 'Alàs i Cerc'
    }
    
    return mapeo_correcciones.get(texto, texto)

# Aplicamos la normalización a la columna de municipios
df['municipio'] = df['municipio'].apply(normalizar_municipio)

# para evitar mostrar datos de municipios con los que no se puede deducir nada, nos quedamos con los municipios que tienen mas de 5 inmubles
# Sacamos los municipios con mas de 5 inmubles y creamos una lista con ellos
frecuencias = df['municipio'].value_counts()
municipios_top = frecuencias[frecuencias > 5].index

# Filtramos el dataframe para quedarnos solo con esos municipios principales
df = df[df['municipio'].isin(municipios_top)]

# Eliminar outliers extremos p.99 para evitar distorsiones visuales, en andorra hay mucho lujo y esto distorsiona mucho las visualizaciones
df = df[df['precio'] < df['precio'].quantile(0.99)]
df = df[df['m2'] < df['m2'].quantile(0.99)]

# Creamos la columna de desviación porcentual respecto a la mediana del mercado local por municipio  y tipo de vivienda
mediana_mercado = df.groupby(['municipio', 'tipo_vivienda'])['eur_m2'].transform('median')
df['desviacion_zona_pct'] = ((df['eur_m2'] - mediana_mercado) / mediana_mercado) * 100

def clasificar_inmueble(desviacion):
    if desviacion <= -15:
        return 'Infravalorado (< -15%)'
    elif desviacion >= 15:
        return 'Sobrevalorado (> +15%)'
    else:
        return 'Precio de Mercado'

df['clasificacion_precio'] = df['desviacion_zona_pct'].apply(clasificar_inmueble)

# Obtener lista única de municipios ordenados para el filtro HTML
lista_municipios = sorted(df['municipio'].unique())

print(f"tras el proceso, el csv tiene {df.shape[0]} registros")

# GENERACIÓN DE LAS 6 VISUALIZACIONES

ALTURA_GRAFICO = 550

# Acto I - G1 - Asimetría de Precios por Territorio
df_municipios = df.groupby('municipio')['eur_m2'].mean().sort_values(ascending=True).reset_index()
fig1 = px.bar(
    df_municipios,
    x='eur_m2',
    y='municipio',
    orientation='h',
    color='eur_m2',
    color_continuous_scale='YlOrRd',
    labels={'eur_m2': 'Precio Medio (€/m2)', 'municipio': 'Municipio / Parroquia'},
    template='plotly_white',
    height=ALTURA_GRAFICO
)
fig1.update_layout(title="<b>Precio Medio por Metro Cuadrado por Municipio / Parroquia</b>", margin=dict(l=150))

# Acto I - G2 - Concentración de Tipologías de Lujo
fig2 = px.density_heatmap(
    df, 
    x="municipio", 
    y="tipo_vivienda", 
    z="precio", 
    histfunc="avg",
    color_continuous_scale="Viridis",
    labels={'municipio': 'Municipio', 'tipo_vivienda': 'Tipo de Vivienda', 'precio': 'Precio Promedio'},
    template='plotly_white',
    height=ALTURA_GRAFICO
)
fig2.update_layout(title="<b>¿Qué tipos de vivienda valen más y dónde?</b>")

# Acto II - G3 - Comparación de Volumen de Inmuebles Activos vs. Vendidos por Rangos de Precio
fig3 = px.histogram(
    df,
    x="eur_m2",
    color="estado",
    barmode="overlay",
    nbins=40,
    color_discrete_map={'Activo': '#3498db', 'Vendido': '#2ecc71'},
    template='plotly_white',
    height=ALTURA_GRAFICO
)
fig3.update_traces(opacity=0.7)
fig3.update_layout(
    title="<b>Volumen de Inmuebles Activos vs. Vendidos por rangos de precio</b>",
    xaxis_title="Precio por metro cuadrado (€/m2)",
    yaxis_title="Cantidad de inmuebles"
)

# GENERACIÓN DE GRÁFICOS MULTI-MUNICIPIO

# Para los gráficos 4, 5 y 6, vamos a generar un gráfico independiente POR CADA MUNICIPIO
# y el HTML se encargará de ocultar/mostrar el correcto según la selección del usuario.
gráficos_4_html = ""
gráficos_5_html = ""
gráficos_6_html = ""

# Primero creamos las versiones "Globales" para cuuando no haya ningun filtro aplicado, es decir, para todos los municipios
fig4_global = px.box(df, x='municipio', y='eur_m2', color='municipio', template='plotly_white', height=ALTURA_GRAFICO)
fig4_global.update_layout(showlegend=False, xaxis_title="Municipio", yaxis_title="Precio por Metro Cuadrado (€/m2)", title="<b>Dispersión de Precios: Todos los Municipios</b>")
gráficos_4_html += f'<div class="grafico-filtrable" data-municipio="todos">{plot(fig4_global, output_type="div", include_plotlyjs=False)}</div>'

fig5_global = px.scatter(df, x='m2', y='precio', color='clasificacion_precio', size='eur_m2', hover_data=['municipio', 'tipo_vivienda', 'eur_m2'], color_discrete_map={'Infravalorado (< -15%)': '#2ecc71','Precio de Mercado': '#f1c40f','Sobrevalorado (> +15%)': '#e74c3c'}, template='plotly_white', height=ALTURA_GRAFICO)
fig5_global.update_layout(title="<b>El Filtro de Oportunidades: Todo el Catálogo de Andorra</b>", xaxis_title="Superficie Construida (m2)", yaxis_title="Precio Total Inmueble (€)", legend_title="Diagnóstico")
gráficos_5_html += f'<div class="grafico-filtrable" data-municipio="todos">{plot(fig5_global, output_type="div", include_plotlyjs=False)}</div>'

df_gangas_global = df[df['clasificacion_precio'] == 'Infravalorado (< -15%)']
fig6_global = px.treemap(df_gangas_global, path=['municipio', 'tipo_vivienda', 'id_inmueble'], values='precio', color='desviacion_zona_pct', color_continuous_scale='Greens_r', template='plotly_white', height=ALTURA_GRAFICO+100)
fig6_global.update_layout(title="<b>Mapa global de inmuebles infravalorados disponibles</b>")
gráficos_6_html += f'<div class="grafico-filtrable" data-municipio="todos">{plot(fig6_global, output_type="div", include_plotlyjs=False)}</div>'

# Generamos los gráficos individuales por cada municipio específico
for muni in lista_municipios:
    df_muni = df[df['municipio'] == muni]
    
    # Iteramos por cada municipio para crear un boxplot específico de dispersión de precios por tipo de vivienda
    fig4 = px.box(df_muni, x='tipo_vivienda', y='eur_m2', color='tipo_vivienda', template='plotly_white', height=ALTURA_GRAFICO)
    fig4.update_layout(showlegend=False, xaxis_title="Tipo de Vivienda", yaxis_title="Precio por Metro Cuadrado (€/m2)", title=f"<b>Dispersión de Precios en {muni} por tipo de vivienda</b>")
    gráficos_4_html += f'<div class="grafico-filtrable" data-municipio="{muni}" style="display:none;">{plot(fig4, output_type="div", include_plotlyjs=False)}</div>'
    
    # Iteramos por cada municipio para crear un scatter plot específico con la clasificación de precios
    fig5 = px.scatter(df_muni, x='m2', y='precio', color='clasificacion_precio', size='eur_m2', hover_data=['tipo_vivienda', 'eur_m2'], color_discrete_map={'Infravalorado (< -15%)': '#2ecc71','Precio de Mercado': '#f1c40f','Sobrevalorado (> +15%)': '#e74c3c'}, template='plotly_white', height=ALTURA_GRAFICO)
    fig5.update_layout(title=f"<b>Oportunidades Localizadas en {muni}</b>", xaxis_title="Superficie Construida (m2)", yaxis_title="Precio Total (€)", legend_title="Diagnóstico")
    gráficos_5_html += f'<div class="grafico-filtrable" data-municipio="{muni}" style="display:none;">{plot(fig5, output_type="div", include_plotlyjs=False)}</div>'
    
    # Iteramos por cada municipio para crear un treemap específico de viviendas infravaloradas disponibles
    df_gangas_muni = df_muni[df_muni['clasificacion_precio'] == 'Infravalorado (< -15%)']
    if not df_gangas_muni.empty:
        fig6 = px.treemap(df_gangas_muni, path=['tipo_vivienda', 'id_inmueble'], values='precio', color='desviacion_zona_pct', color_continuous_scale='Greens_r', template='plotly_white', height=ALTURA_GRAFICO+100)
        fig6.update_layout(title=f"<b>Inventario de viviendas infravaloradas en {muni}</b>")
        gráficos_6_html += f'<div class="grafico-filtrable" data-municipio="{muni}" style="display:none;">{plot(fig6, output_type="div", include_plotlyjs=False)}</div>'
    else:
        # Si no hay gangas en ese municipio, ponemos un aviso amigable en HTML
        gráficos_6_html += f'<div class="grafico-filtrable aviso-no-datos" data-municipio="{muni}" style="display:none; padding:40px; text-align:center; background:#fef2f2; border:1px solid #fee2e2; border-radius:12px; color:#991b1b;">⚠️ Actualmente no se detectan inmuebles bajo el criterio matemático de "Ganga" activo en el municipio de {muni}.</div>'


# =========================================================
# CONSTRUCCIÓN DEL DASHBOARD HTML COMPLETO
# =========================================================

options_html = '<option value="todos"> Ver Todo Andorra</option>'
for muni in lista_municipios:
    options_html += f'<option value="{muni}"> {muni}</option>'

html_string = f'''
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard de Análisis Inmobiliario</title>
    
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap" rel="stylesheet">
    
    <style>
        :root {{
            --bg-principal: #f8fafc;
            --bg-tarjeta: #ffffff;
            --texto-principal: #1e293b;
            --texto-secundario: #64748b;
            --borde: #e2e8f0;
            --azul-marca: #1e3a8a;
            --azul-interactivo: #3b82f6;
        }}
        
        body {{
            font-family: 'Inter', sans-serif;
            background-color: var(--bg-principal);
            color: var(--texto-principal);
            margin: 0;
            padding: 0;
            line-height: 1.6;
        }}
        
        .header-narrativo {{
            background: linear-gradient(135deg, #1e3a8a 0%, #0f172a 100%);
            color: white;
            padding: 60px 20px;
            text-align: center;
            box-shadow: 0 4px 20px rgba(0,0,0,0.08);
        }}
        
        .header-narrativo h1 {{
            font-size: 2.3rem;
            margin: 0 0 15px 0;
            font-weight: 700;
            letter-spacing: -0.03em;
        }}
        
        .header-narrativo p {{
            font-size: 1.1rem;
            max-width: 800px;
            margin: 0 auto;
            color: #94a3b8;
            font-weight: 300;
        }}
        
        .contenedor-storytelling {{
            max-width: 1000px;
            margin: 40px auto;
            padding: 0 20px;
        }}
        
        .acto-separador {{
            margin-top: 60px;
            margin-bottom: 25px;
            border-bottom: 2px solid #e2e8f0;
            padding-bottom: 10px;
        }}
        
        .acto-titulo {{
            font-size: 1.4rem;
            color: var(--azul-marca);
            font-weight: 700;
            text-transform: uppercase;
            letter-spacing: 0.05em;
        }}
        
        /* CONTENEDOR FLOTANTE E INTUITIVO DEL FILTRO */
        .contenedor-filtro-fijo {{
            background: #ffffff;
            border: 2px solid var(--azul-interactivo);
            border-radius: 12px;
            padding: 20px;
            margin: 30px 0;
            box-shadow: 0 10px 15px -3px rgba(59, 130, 246, 0.1);
            display: flex;
            align-items: center;
            justify-content: space-between;
            flex-wrap: wrap;
            gap: 15px;
        }}
        
        .filtro-texto h4 {{
            margin: 0;
            font-size: 1.1rem;
            color: var(--azul-marca);
        }}
        
        .filtro-texto p {{
            margin: 2px 0 0 0;
            font-size: 0.85rem;
            color: var(--texto-secundario);
        }}
        
        .selector-municipio {{
            font-family: 'Inter', sans-serif;
            font-size: 1rem;
            font-weight: 600;
            padding: 10px 20px;
            border-radius: 8px;
            border: 1px solid var(--borde);
            background-color: #ffffff;
            color: var(--texto-principal);
            cursor: pointer;
            outline: none;
            box-shadow: 0 1px 3px rgba(0,0,0,0.05);
            transition: all 0.2s ease;
            min-width: 280px;
        }}
        
        .selector-municipio:focus {{
            border-color: var(--azul-interactivo);
            box-shadow: 0 0 0 3px rgba(59, 130, 246, 0.15);
        }}
        
        .tarjeta-grafico-lineal {{
            background: var(--bg-tarjeta);
            border: 1px solid var(--borde);
            border-radius: 16px;
            padding: 30px;
            margin-bottom: 40px;
            box-shadow: 0 4px 10px rgba(0, 0, 0, 0.03);
        }}
        
        .tarjeta-grafico-lineal h3 {{
            font-size: 1.3rem;
            margin-top: 0;
            margin-bottom: 12px;
            color: var(--texto-principal);
            font-weight: 600;
        }}
        
        .tarjeta-grafico-lineal p {{
            font-size: 1rem;
            color: var(--texto-secundario);
            margin-bottom: 25px;
            max-width: 900px;
        }}
        
        footer {{
            text-align: center;
            padding: 40px 20px;
            background: #0f172a;
            color: #64748b;
            font-size: 0.9rem;
            margin-top: 80px;
        }}
    </style>
</head>
<body>

<div class="header-narrativo">
    <h1>¿Cómo encontrar una oportunidades inmobiliarias en el mercado inmobiliario andorrano?</h1>
    <p>Guía para identificar el valor real de un inmueble en Andorra, analizando su tipología, ubicación y precio para detectar oportunidades y evitar propiedades sobrevaloradas.</p>
</div>

<div class="contenedor-storytelling">

    <div class="acto-separador">
        <div class="acto-titulo">PARTE I: Analizando la zona</div>
    </div>

    <div class="tarjeta-grafico-lineal">
        <h3>1. Asimetría de precios por territorio</h3>
        <p>El primer desafío es entender que Andorra no es homogénea. Este gráfico permite analizar de forma aislada qué municipios registran los costes por metro cuadrado más prohibitivos, permitiendo identificar las zonas de mayor y menor accesibilidad.</p>
        {plot(fig1, output_type='div', include_plotlyjs='cdn')}
    </div>
    
    <div class="tarjeta-grafico-lineal">
        <h3>2. Cruza de variables, ¿Lo que buscamos dónde lo encontramos?</h3>
        <p>Analizamos las zonas teniendo en cuenta el tipo de vivienda. Este análisis nos permite identificar patrones y tendencias que no serían evidentes al observar cada variable por separado.</p>
        {plot(fig2, output_type='div', include_plotlyjs=False)}
    </div>


    <div class="acto-separador">
        <div class="acto-titulo">PARTE II: El tiempo determina lo que es un inmueble sobrevalorado</div>
    </div>

    <div class="tarjeta-grafico-lineal">
        <h3>3. Comparativa de precios de inmuebles activos vs. vendidos</h3>
        <p>Los portales web son estáticos, pero nuestro histórico acumulado revela qué inmuebles se venden rápido y cuáles se estancan. Al superponer los inmuebles efectivamente vendidos sobre el stock activo por rangos de precio, evidenciamos gráficamente si el volumen de mercado acumulado sufre de un problema de sobrevaloración y en que rangos de precio ocurre más frecuentemente.</p>
        {plot(fig3, output_type='div', include_plotlyjs=False)}
    </div>


    <div class="contenedor-filtro-fijo">
        <div class="filtro-texto">
            <h4> Centro de Comando del Comprador</h4>
            <p>Selecciona un municipio para filtrar dinámicamente el/los municipios a analizary para los graficos 4, 5 y 6.</p>
        </div>
        <select id="selectorMuni" class="selector-municipio" onchange="filtrarTableros()">
            {options_html}
        </select>
    </div>


    <div class="acto-separador">
        <div class="acto-titulo">PARTE III: La Definición Estadística de "Infravaloración"</div>
    </div>

    <div class="tarjeta-grafico-lineal" id="bloque-grafico-4">
        <h3>4. Rangos y Dispersión del Mercado Local</h3>
        <p>Un inmueble se clasifica como infravalorado cuando su precio es más de un 15% inferior al valor de mercado estimado para su zona. En términos visuales, estas propiedades suelen concentrarse en la parte baja del boxplot y, en los casos más extremos, aparecen como valores atípicos inferiores. Cuanto más alejada esté una vivienda de la distribución central de precios, mayor es la evidencia de que podría estar ofreciendo un valor superior al esperado para su precio.</p>
        {gráficos_4_html}
    </div>
    
    <div class="tarjeta-grafico-lineal" id="bloque-grafico-5">
        <h3>5. Diagnóstico automático del mercado local</h3>
        <p>Para visualizar las oportunidades en una zona, las propiedades con un precio inferior al 15% de la mediana de su mercado local se marcan automáticamente en <b>Verde</b>. El gráfic muestra aquellas propiedades que se encuentran en la media del mercado (+-15% del precio mediano) de color amarillo y las propiedades sobrevaloradas (+15% del precio mediano) de color rojo.</p>
        {gráficos_5_html}
    </div>


    <div class="acto-separador">
        <div class="acto-titulo">PARTE IV: Captura de la oportunidad</div>
    </div>

    <div class="tarjeta-grafico-lineal" id="bloque-grafico-6">
        <h3>6. Mapa de navegación de oportunidades infravaloradas activas</h3>
        <p>Esta visualización muestra exclusivamente las propiedades infravaloradas que continúan disponibles en el mercado (estado "Activo"), eliminando el resto de inmuebles para facilitar el análisis. De esta forma, el usuario puede centrarse únicamente en aquellas viviendas cuyo precio se encuentra significativamente por debajo del valor estimado para su zona.

Al seleccionar una región concreta, es posible identificar el tipo de inmueble, consultar su identificador y comparar el porcentaje de infravaloración estimado, facilitando la localización de oportunidades de compra potencialmente atractivas.</p>
        {gráficos_6_html}
    </div>

</div>

<footer>
    Proyecto de Visualización de Datos — Análisis del Mercado Inmobiliario de Andorra. Desarrollado con Plotly y Filtros Avanzados JavaScript por Rodrigo Cacace Fontalva.
</footer>

<script>
function filtrarTableros() {{
    var municipioSeleccionado = document.getElementById("selectorMuni").value;
    
    // Lista de contenedores de gráficos a los que aplica el filtro
    var graficos = document.getElementsByClassName("grafico-filtrable");
    
    for (var i = 0; i < graficos.length; i++) {{
        var grafico = graficos[i];
        var municipioGrafico = grafico.getAttribute("data-municipio");
        
        if (municipioGrafico === municipioSeleccionado) {{
            grafico.style.display = "block";
            // Forzar a Plotly a redibujar y ajustar tamaños internos en caliente
            var plotlyDiv = grafico.querySelector(".plotly-graph-div");
            if(plotlyDiv && window.Plotly) {{
                Plotly.Plots.resize(plotlyDiv);
            }}
        }} else {{
            grafico.style.display = "none";
        }}
    }}
}}
</script>

</body>
</html>
'''

# guardamos el html generado en un archivo para poder abrirlo en el navegador
with open("proyecto_visualizacion_andorra.html", "w", encoding="utf-8") as f:
    f.write(html_string)

print("HTML generado exitosamente: proyecto_visualizacion_andorra.html")

el csv tiene 2140 registros
tras el proceso, el csv tiene 2012 registros
HTML generado exitosamente: proyecto_visualizacion_andorra.html
